# 05 - Evaluation

Compare base model vs fine-tuned model on test set.

**Prerequisites:**
- Run `04_training.ipynb` first
- Training output must be available (LoRA adapter)

**Runtime:** Kaggle T4 GPU, ~20-30 minutes

In [ ]:
%%capture
!pip install unsloth rouge_score bert_score

In [ ]:
import os
import sys
import json
import time

os.chdir("/kaggle/working")
if not os.path.exists("fingpt-qlora"):
    !git clone https://github.com/W-Kaski/fingpt-qlora.git
os.chdir("fingpt-qlora")

# Setup logging
log_path = "results/evaluation_log.txt"
os.makedirs("results", exist_ok=True)

class Tee:
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()

log_file = open(log_path, "w")
sys.stdout = Tee(sys.__stdout__, log_file)
sys.stderr = Tee(sys.__stderr__, log_file)

def log(msg):
    timestamp = time.strftime("%H:%M:%S")
    print(f"[{timestamp}] {msg}")

log("Evaluation started")

In [ ]:
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# Load config
with open("configs/base.yaml") as f:
    config = yaml.safe_load(f)

EXPERIMENT_NAME = "v1_baseline"
MODEL_NAME = config["model"]["name"]
MAX_SEQ_LENGTH = config["model"]["max_seq_length"]

log(f"Config loaded:")
log(f"  Model: {MODEL_NAME}")
log(f"  Max seq length: {MAX_SEQ_LENGTH}")
log(f"  Experiment: {EXPERIMENT_NAME}")

# Load test data
with open("data/splits/test.json") as f:
    test_data = json.load(f)

log(f"\nTest set: {len(test_data)} examples")
task_counts = Counter(ex.get("task_type", "unknown") for ex in test_data)
for task, count in sorted(task_counts.items()):
    log(f"  {task}: {count}")

In [ ]:
from unsloth import FastLanguageModel

def generate_response(model, tokenizer, messages, max_new_tokens=512):
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            top_p=0.9,
            do_sample=True,
        )
    return tokenizer.decode(output_ids[0][input_ids.shape[-1]:], skip_special_tokens=True)

def build_messages(conversations):
    messages = []
    for turn in conversations:
        role = turn["from"]
        if role == "gpt":
            role = "assistant"
        messages.append({"role": role, "content": turn["value"]})
    return messages[:-1]

log("Helper functions defined.")

In [ ]:
log("\n=== Loading Base Model ===")
t0 = time.time()

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

t1 = time.time()
log(f"Base model loaded in {t1-t0:.1f}s")
log(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Generate base predictions
log("\n=== Generating Base Predictions ===")
base_predictions = []
t0 = time.time()

for i, example in enumerate(test_data):
    messages = build_messages(example["conversations"])
    pred = generate_response(base_model, base_tokenizer, messages)
    base_predictions.append(pred)
    if (i + 1) % 100 == 0:
        log(f"  Base model: {i+1}/{len(test_data)}")

t1 = time.time()
log(f"Base predictions done: {len(base_predictions)} in {t1-t0:.1f}s")

# Free GPU memory
del base_model
del base_tokenizer
torch.cuda.empty_cache()
import gc; gc.collect()
log("GPU memory freed.")

In [ ]:
log("\n=== Loading Fine-tuned Model ===")
ft_model_path = f"outputs/{EXPERIMENT_NAME}/lora_adapter"
log(f"Loading from: {ft_model_path}")
t0 = time.time()

ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=ft_model_path,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(ft_model)

t1 = time.time()
log(f"Fine-tuned model loaded in {t1-t0:.1f}s")
log(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Generate fine-tuned predictions
log("\n=== Generating Fine-tuned Predictions ===")
ft_predictions = []
t0 = time.time()

for i, example in enumerate(test_data):
    messages = build_messages(example["conversations"])
    pred = generate_response(ft_model, ft_tokenizer, messages)
    ft_predictions.append(pred)
    if (i + 1) % 100 == 0:
        log(f"  Fine-tuned: {i+1}/{len(test_data)}")

t1 = time.time()
log(f"Fine-tuned predictions done: {len(ft_predictions)} in {t1-t0:.1f}s")

# Free GPU memory
del ft_model
del ft_tokenizer
torch.cuda.empty_cache()
gc.collect()
log("GPU memory freed.")

# Assemble results
results = []
for i, example in enumerate(test_data):
    results.append({
        "reference": example["conversations"][-1]["value"],
        "base_prediction": base_predictions[i],
        "ft_prediction": ft_predictions[i],
        "task_type": example.get("task_type", "unknown"),
    })

os.makedirs("results/eval_outputs", exist_ok=True)
with open(f"results/eval_outputs/{EXPERIMENT_NAME}.json", "w") as f:
    json.dump(results, f, indent=2)
log(f"\nSaved {len(results)} results to results/eval_outputs/{EXPERIMENT_NAME}.json")

In [ ]:
import re

def extract_sentiment(text):
    match = re.search(r"\*\*Sentiment:\s*(\w+)\*\*", text, re.IGNORECASE)
    if match:
        label = match.group(1).capitalize()
        if label in ("Positive", "Negative", "Neutral"):
            return label
    text_lower = text.lower()
    if "positive" in text_lower and "negative" not in text_lower:
        return "Positive"
    if "negative" in text_lower and "positive" not in text_lower:
        return "Negative"
    return "Neutral"

def compute_sentiment_metrics(predictions, references):
    pred_labels = [extract_sentiment(p) for p in predictions]
    ref_labels = [extract_sentiment(r) for r in references]
    correct = sum(1 for p, r in zip(pred_labels, ref_labels) if p == r)
    return {"accuracy": correct / len(ref_labels), "correct": correct, "total": len(ref_labels)}

def compute_rouge_l(predictions, references):
    from rouge_score import rouge_scorer
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scores = [scorer.score(r, p)["rougeL"].fmeasure for p, r in zip(predictions, references)]
    return {"rouge_l_f1": np.mean(scores), "std": np.std(scores)}

log("Metric functions defined.")

In [ ]:
log("\n=== Computing Metrics ===")

metrics_table = []

for task_type in sorted(set(r["task_type"] for r in results)):
    task_results = [r for r in results if r["task_type"] == task_type]
    if not task_results:
        continue
    
    refs = [r["reference"] for r in task_results]
    base_preds = [r["base_prediction"] for r in task_results]
    ft_preds = [r["ft_prediction"] for r in task_results]
    
    if task_type == "sentiment":
        base_m = compute_sentiment_metrics(base_preds, refs)
        ft_m = compute_sentiment_metrics(ft_preds, refs)
        metrics_table.append({
            "Task": task_type,
            "Metric": "Accuracy",
            "Base": f"{base_m['accuracy']:.3f}",
            "Fine-tuned": f"{ft_m['accuracy']:.3f}",
            "Delta": f"{ft_m['accuracy'] - base_m['accuracy']:+.3f}",
        })
    
    # ROUGE-L for all tasks
    try:
        base_rouge = compute_rouge_l(base_preds, refs)
        ft_rouge = compute_rouge_l(ft_preds, refs)
        metrics_table.append({
            "Task": task_type,
            "Metric": "ROUGE-L",
            "Base": f"{base_rouge['rouge_l_f1']:.3f}",
            "Fine-tuned": f"{ft_rouge['rouge_l_f1']:.3f}",
            "Delta": f"{ft_rouge['rouge_l_f1'] - base_rouge['rouge_l_f1']:+.3f}",
        })
    except Exception as e:
        log(f"ROUGE-L failed for {task_type}: {e}")

df_metrics = pd.DataFrame(metrics_table)
log("\n" + "="*70)
log("EVALUATION RESULTS")
log("="*70)
log(df_metrics.to_string(index=False))

# Save
df_metrics.to_markdown("results/comparison_table.md", index=False)
log(f"\nSaved to results/comparison_table.md")

In [ ]:
log("\n=== Visualizing Results ===")

os.makedirs("results/figures", exist_ok=True)

fig, ax = plt.subplots(figsize=(10, 6))

sentiment_metrics = [m for m in metrics_table if m["Metric"] == "Accuracy"]
if sentiment_metrics:
    labels = [m["Task"] for m in sentiment_metrics]
    base_vals = [float(m["Base"]) for m in sentiment_metrics]
    ft_vals = [float(m["Fine-tuned"]) for m in sentiment_metrics]
    
    x = np.arange(len(labels))
    width = 0.35
    ax.bar(x - width/2, base_vals, width, label="Base Model", color="#e74c3c", alpha=0.8)
    ax.bar(x + width/2, ft_vals, width, label="Fine-tuned", color="#2ecc71", alpha=0.8)
    ax.set_ylabel("Score")
    ax.set_title(f"Model Comparison - {EXPERIMENT_NAME}")
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()
    ax.set_ylim(0, 1)
    
    for i, (b, f) in enumerate(zip(base_vals, ft_vals)):
        ax.text(i - width/2, b + 0.01, f"{b:.2f}", ha="center", fontsize=9)
        ax.text(i + width/2, f + 0.01, f"{f:.2f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(f"results/figures/comparison_{EXPERIMENT_NAME}.png", dpi=150)
plt.show()
log(f"Chart saved to results/figures/comparison_{EXPERIMENT_NAME}.png")

In [ ]:
log("\n=== Example Outputs ===")

for task_type in sorted(set(r["task_type"] for r in results)):
    task_results = [r for r in results if r["task_type"] == task_type][:3]
    
    log(f"\n{'='*70}")
    log(f"Task: {task_type}")
    log(f"{'='*70}")
    
    for i, r in enumerate(task_results):
        log(f"\n--- Example {i+1} ---")
        log(f"Reference: {r['reference'][:200]}...")
        log(f"\nBase Model:\n{r['base_prediction'][:300]}...")
        log(f"\nFine-tuned:\n{r['ft_prediction'][:300]}...")

In [ ]:
log("\n=== Bootstrap Confidence Intervals ===")

sentiment_results = [r for r in results if r["task_type"] == "sentiment"]
if sentiment_results:
    refs = [r["reference"] for r in sentiment_results]
    ft_preds = [r["ft_prediction"] for r in sentiment_results]
    
    rng = np.random.RandomState(42)
    n = len(refs)
    boot_scores = []
    
    for _ in range(1000):
        idx = rng.choice(n, size=n, replace=True)
        boot_refs = [refs[i] for i in idx]
        boot_preds = [ft_preds[i] for i in idx]
        m = compute_sentiment_metrics(boot_preds, boot_refs)
        boot_scores.append(m["accuracy"])
    
    ci_lower = np.percentile(boot_scores, 2.5)
    ci_upper = np.percentile(boot_scores, 97.5)
    
    log(f"\nSentiment Accuracy (Fine-tuned):")
    log(f"  Mean: {np.mean(boot_scores):.3f}")
    log(f"  95% CI: [{ci_lower:.3f}, {ci_upper:.3f}]")
    log(f"  Std: {np.std(boot_scores):.3f}")
else:
    log("No sentiment examples in test set.")

log("\n" + "="*70)
log("EVALUATION COMPLETE")
log("="*70)
log(f"\nLog saved to: {log_path}")